# iCHO3K Model - Batch Simulation and Metabolite Combinations

This notebook enables you to:
1. Load and run the iCHO3K genome-scale metabolic model for CHO cells
2. Simulate batch cultures over multiple days
3. Test combinations of metabolites and their effects on cell growth

**Model Details:**
- 11,004 reactions
- 7,377 metabolites
- 3,597 genes

**Reference:** Di Giusto et al. (2025), bioRxiv. https://doi.org/10.1101/2025.04.10.647063

## 1. Setup and Installation

Install required dependencies for metabolic modeling with COBRApy

In [ ]:
# Install dependencies
!pip install cobra python-libsbml optlang swiglpk pandas numpy scipy matplotlib seaborn tqdm openpyxl -q

print("✓ Dependencies installed successfully!")

## 2. Clone iCHO3K Repository

Download the iCHO3K model from GitHub

In [ ]:
import os

# Clone repository if not already present
if not os.path.exists('iCHO3K'):
    !git clone https://github.com/LewisLabUCSD/iCHO3K.git
    print("✓ Repository cloned successfully!")
else:
    print("✓ Repository already exists!")

# List model files
!ls -lh iCHO3K/iCHO3K/Model/*.json

## 3. Import Libraries

In [ ]:
import cobra
from cobra.io import load_json_model
from cobra.flux_analysis import pfba
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ Libraries imported successfully!")
print(f"COBRApy version: {cobra.__version__}")

## 4. Load iCHO3K Model

In [ ]:
# Load the production CHO model (unblocked version)
MODEL_PATH = 'iCHO3K/iCHO3K/Model/iCHO3K_cho_prod_generic_unblocked.json'

print("Loading iCHO3K model...")
model = load_json_model(MODEL_PATH)

# Configure solver
cobra.Configuration().solver = "glpk"  # Free solver, can use 'gurobi' if available

# Set objective to production biomass
model.objective = 'biomass_cho_prod'

print(f"✓ Model loaded successfully!")
print(f"\nModel statistics:")
print(f"  Reactions: {len(model.reactions)}")
print(f"  Metabolites: {len(model.metabolites)}")
print(f"  Genes: {len(model.genes)}")
print(f"  Objective: {model.objective.expression}")

## 5. Test Basic Model Functionality

In [ ]:
# Run basic FBA optimization
print("Running Flux Balance Analysis...")
solution = model.optimize()

if solution.status == 'optimal':
    print(f"✓ Optimization successful!")
    print(f"\nGrowth rate: {solution.objective_value:.4f} h⁻¹")
    
    # Show key exchange fluxes
    print("\nKey exchange reactions:")
    key_exchanges = ['EX_glc__D_e', 'EX_gln__L_e', 'EX_lac__L_e', 'EX_o2_e']
    for rxn_id in key_exchanges:
        if rxn_id in model.reactions:
            flux = solution.fluxes[rxn_id]
            print(f"  {rxn_id}: {flux:.4f} mmol/gDW/h")
else:
    print(f"✗ Optimization failed with status: {solution.status}")

## 6. Setup Minimal Medium and Batch Simulation

First, we'll define a helper function to set up the minimal medium with all essential nutrients

In [ ]:
def simulate_batch_culture(model, 
                          days=7,
                          initial_biomass=0.1,  # g/L
                          initial_glucose=25.0,  # mM
                          initial_glutamine=4.0,  # mM
                          volume=1.0,  # L
                          timestep=1.0,  # hours
                          max_uptake_glucose=10.0,  # mmol/gDW/h
                          max_uptake_glutamine=2.0,  # mmol/gDW/h
                          max_oxygen=20.0):  # mmol/gDW/h
    """
    Simulate batch culture of CHO cells over multiple days.
    
    Parameters:
    -----------
    model : cobra.Model
        The iCHO3K metabolic model
    days : int
        Number of days to simulate
    initial_biomass : float
        Initial biomass concentration (g/L)
    initial_glucose : float
        Initial glucose concentration (mM)
    initial_glutamine : float
        Initial glutamine concentration (mM)
    volume : float
        Culture volume (L)
    timestep : float
        Time step for simulation (hours)
    max_uptake_glucose : float
        Maximum glucose uptake rate (mmol/gDW/h)
    max_uptake_glutamine : float
        Maximum glutamine uptake rate (mmol/gDW/h)
    max_oxygen : float
        Maximum oxygen uptake rate (mmol/gDW/h)
    
    Returns:
    --------
    pd.DataFrame
        Time-course data with biomass, metabolites, and growth rate
    """
    
    # Initialize state variables
    time_hours = []
    biomass = []
    glucose = []
    glutamine = []
    lactate = []
    growth_rates = []
    
    # Set initial conditions
    current_biomass = initial_biomass  # g/L
    current_glucose = initial_glucose  # mM
    current_glutamine = initial_glutamine  # mM
    current_lactate = 0.0  # mM
    
    # Total hours to simulate
    total_hours = days * 24
    num_steps = int(total_hours / timestep)
    
    print(f"Simulating batch culture for {days} days ({total_hours} hours)...")
    
    for step in tqdm(range(num_steps), desc="Simulation progress"):
        current_time = step * timestep
        
        # Create a temporary model copy
        with model:
            # CRITICAL: Set up minimal medium with all essential nutrients
            setup_minimal_medium(model)
            
            # Set metabolite uptake constraints based on availability
            # Glucose uptake (negative = uptake)
            if current_glucose > 0.1:
                glc_uptake = min(max_uptake_glucose, current_glucose * 0.5)
                model.reactions.EX_glc__D_e.bounds = (-glc_uptake, 1000)
            else:
                model.reactions.EX_glc__D_e.bounds = (0, 1000)
            
            # Glutamine uptake
            if current_glutamine > 0.1:
                gln_uptake = min(max_uptake_glutamine, current_glutamine * 0.5)
                model.reactions.EX_gln__L_e.bounds = (-gln_uptake, 1000)
            else:
                model.reactions.EX_gln__L_e.bounds = (0, 1000)
            
            # Oxygen (always available)
            model.reactions.EX_o2_e.bounds = (-max_oxygen, 1000)
            
            # Lactate secretion (positive = secretion)
            model.reactions.EX_lac__L_e.bounds = (0, 1000)
            
            # Run FBA
            try:
                solution = model.optimize()
                
                if solution.status == 'optimal':
                    growth_rate = solution.objective_value
                    
                    # Get metabolite fluxes
                    glc_flux = solution.fluxes['EX_glc__D_e']
                    gln_flux = solution.fluxes['EX_gln__L_e']
                    lac_flux = solution.fluxes['EX_lac__L_e']
                    
                    # Update biomass (exponential growth)
                    biomass_growth = current_biomass * growth_rate * timestep
                    current_biomass += biomass_growth
                    
                    # Update metabolite concentrations
                    # Convert from mmol/gDW/h to mM change in medium
                    total_biomass_g = current_biomass * volume
                    
                    current_glucose += (glc_flux * total_biomass_g * timestep) / volume
                    current_glutamine += (gln_flux * total_biomass_g * timestep) / volume
                    current_lactate += (lac_flux * total_biomass_g * timestep) / volume
                    
                    # Ensure non-negative concentrations
                    current_glucose = max(0, current_glucose)
                    current_glutamine = max(0, current_glutamine)
                    current_lactate = max(0, current_lactate)
                    
                else:
                    growth_rate = 0
                    
            except Exception as e:
                print(f"\nWarning: Optimization failed at t={current_time}h: {e}")
                growth_rate = 0
        
        # Store results
        time_hours.append(current_time)
        biomass.append(current_biomass)
        glucose.append(current_glucose)
        glutamine.append(current_glutamine)
        lactate.append(current_lactate)
        growth_rates.append(growth_rate)
        
        # Stop if biomass drops (cell death) or glucose depleted
        if current_biomass < 0.01 or (current_glucose < 0.01 and current_glutamine < 0.01):
            print(f"\nSimulation stopped at {current_time:.1f}h (nutrient depletion or cell death)")
            break
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'Time (h)': time_hours,
        'Time (days)': [t/24 for t in time_hours],
        'Biomass (g/L)': biomass,
        'Glucose (mM)': glucose,
        'Glutamine (mM)': glutamine,
        'Lactate (mM)': lactate,
        'Growth Rate (1/h)': growth_rates
    })
    
    print(f"\n✓ Batch simulation completed!")
    print(f"Final biomass: {current_biomass:.2f} g/L")
    print(f"Final glucose: {current_glucose:.2f} mM")
    print(f"Final lactate: {current_lactate:.2f} mM")
    
    return results_df

In [ ]:
def simulate_batch_culture(model, 
                          days=7,
                          initial_biomass=0.1,  # g/L
                          initial_glucose=25.0,  # mM
                          initial_glutamine=4.0,  # mM
                          volume=1.0,  # L
                          timestep=1.0,  # hours
                          max_uptake_glucose=10.0,  # mmol/gDW/h
                          max_uptake_glutamine=2.0,  # mmol/gDW/h
                          max_oxygen=20.0):  # mmol/gDW/h
    """
    Simulate batch culture of CHO cells over multiple days.
    
    Parameters:
    -----------
    model : cobra.Model
        The iCHO3K metabolic model
    days : int
        Number of days to simulate
    initial_biomass : float
        Initial biomass concentration (g/L)
    initial_glucose : float
        Initial glucose concentration (mM)
    initial_glutamine : float
        Initial glutamine concentration (mM)
    volume : float
        Culture volume (L)
    timestep : float
        Time step for simulation (hours)
    max_uptake_glucose : float
        Maximum glucose uptake rate (mmol/gDW/h)
    max_uptake_glutamine : float
        Maximum glutamine uptake rate (mmol/gDW/h)
    max_oxygen : float
        Maximum oxygen uptake rate (mmol/gDW/h)
    
    Returns:
    --------
    pd.DataFrame
        Time-course data with biomass, metabolites, and growth rate
    """
    
    # Initialize state variables
    time_hours = []
    biomass = []
    glucose = []
    glutamine = []
    lactate = []
    growth_rates = []
    
    # Set initial conditions
    current_biomass = initial_biomass  # g/L
    current_glucose = initial_glucose  # mM
    current_glutamine = initial_glutamine  # mM
    current_lactate = 0.0  # mM
    
    # Total hours to simulate
    total_hours = days * 24
    num_steps = int(total_hours / timestep)
    
    print(f"Simulating batch culture for {days} days ({total_hours} hours)...")
    
    for step in tqdm(range(num_steps), desc="Simulation progress"):
        current_time = step * timestep
        
        # Create a temporary model copy
        with model:
            # Set metabolite uptake constraints based on availability
            # Glucose uptake (negative = uptake)
            if current_glucose > 0.1:
                glc_uptake = min(max_uptake_glucose, current_glucose * 0.5)
                model.reactions.EX_glc__D_e.bounds = (-glc_uptake, 0)
            else:
                model.reactions.EX_glc__D_e.bounds = (0, 0)
            
            # Glutamine uptake
            if current_glutamine > 0.1:
                gln_uptake = min(max_uptake_glutamine, current_glutamine * 0.5)
                model.reactions.EX_gln__L_e.bounds = (-gln_uptake, 0)
            else:
                model.reactions.EX_gln__L_e.bounds = (0, 0)
            
            # Oxygen (always available)
            model.reactions.EX_o2_e.bounds = (-max_oxygen, 0)
            
            # Lactate secretion (positive = secretion)
            model.reactions.EX_lac__L_e.bounds = (0, 1000)
            
            # Run FBA
            try:
                solution = model.optimize()
                
                if solution.status == 'optimal':
                    growth_rate = solution.objective_value
                    
                    # Get metabolite fluxes
                    glc_flux = solution.fluxes['EX_glc__D_e']
                    gln_flux = solution.fluxes['EX_gln__L_e']
                    lac_flux = solution.fluxes['EX_lac__L_e']
                    
                    # Update biomass (exponential growth)
                    biomass_growth = current_biomass * growth_rate * timestep
                    current_biomass += biomass_growth
                    
                    # Update metabolite concentrations
                    # Convert from mmol/gDW/h to mM change in medium
                    total_biomass_g = current_biomass * volume
                    
                    current_glucose += (glc_flux * total_biomass_g * timestep) / volume
                    current_glutamine += (gln_flux * total_biomass_g * timestep) / volume
                    current_lactate += (lac_flux * total_biomass_g * timestep) / volume
                    
                    # Ensure non-negative concentrations
                    current_glucose = max(0, current_glucose)
                    current_glutamine = max(0, current_glutamine)
                    current_lactate = max(0, current_lactate)
                    
                else:
                    growth_rate = 0
                    
            except Exception as e:
                print(f"Warning: Optimization failed at t={current_time}h: {e}")
                growth_rate = 0
        
        # Store results
        time_hours.append(current_time)
        biomass.append(current_biomass)
        glucose.append(current_glucose)
        glutamine.append(current_glutamine)
        lactate.append(current_lactate)
        growth_rates.append(growth_rate)
        
        # Stop if biomass drops (cell death) or glucose depleted
        if current_biomass < 0.01 or (current_glucose < 0.01 and current_glutamine < 0.01):
            print(f"\nSimulation stopped at {current_time:.1f}h (nutrient depletion or cell death)")
            break
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'Time (h)': time_hours,
        'Time (days)': [t/24 for t in time_hours],
        'Biomass (g/L)': biomass,
        'Glucose (mM)': glucose,
        'Glutamine (mM)': glutamine,
        'Lactate (mM)': lactate,
        'Growth Rate (1/h)': growth_rates
    })
    
    print(f"\n✓ Batch simulation completed!")
    print(f"Final biomass: {current_biomass:.2f} g/L")
    print(f"Final glucose: {current_glucose:.2f} mM")
    print(f"Final lactate: {current_lactate:.2f} mM")
    
    return results_df

### 6.1 Run Batch Simulation

Simulate a 7-day batch culture

In [ ]:
# Run batch simulation
batch_results = simulate_batch_culture(
    model=model,
    days=7,
    initial_biomass=0.2,
    initial_glucose=25.0,
    initial_glutamine=4.0,
    timestep=1.0
)

# Display results
print("\nBatch simulation results (first 10 rows):")
print(batch_results.head(10))

### 6.2 Visualize Batch Culture Results

In [ ]:
def plot_batch_results(df):
    """
    Create comprehensive plots of batch culture simulation results.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Biomass growth
    axes[0, 0].plot(df['Time (days)'], df['Biomass (g/L)'], 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Time (days)', fontsize=11)
    axes[0, 0].set_ylabel('Biomass (g/L)', fontsize=11)
    axes[0, 0].set_title('Cell Growth', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Glucose and Glutamine
    ax2 = axes[0, 1]
    ax2.plot(df['Time (days)'], df['Glucose (mM)'], 'g-', linewidth=2, label='Glucose')
    ax2.plot(df['Time (days)'], df['Glutamine (mM)'], 'orange', linewidth=2, label='Glutamine')
    ax2.set_xlabel('Time (days)', fontsize=11)
    ax2.set_ylabel('Concentration (mM)', fontsize=11)
    ax2.set_title('Nutrient Consumption', fontsize=12, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Lactate production
    axes[1, 0].plot(df['Time (days)'], df['Lactate (mM)'], 'r-', linewidth=2)
    axes[1, 0].set_xlabel('Time (days)', fontsize=11)
    axes[1, 0].set_ylabel('Lactate (mM)', fontsize=11)
    axes[1, 0].set_title('Lactate Production', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Growth rate
    axes[1, 1].plot(df['Time (days)'], df['Growth Rate (1/h)'], 'purple', linewidth=2)
    axes[1, 1].set_xlabel('Time (days)', fontsize=11)
    axes[1, 1].set_ylabel('Growth Rate (1/h)', fontsize=11)
    axes[1, 1].set_title('Specific Growth Rate', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot results
plot_batch_results(batch_results)

def analyze_metabolite_combinations(model, 
                                   metabolites_dict,
                                   combination_size=2,
                                   base_metabolites=None):
    """
    Analyze the effect of metabolite combinations on growth.
    
    Parameters:
    -----------
    model : cobra.Model
        The iCHO3K metabolic model
    metabolites_dict : dict
        Dictionary mapping metabolite names to exchange reaction IDs and uptake rates
        Example: {'Glucose': ('EX_glc__D_e', -10), 'Glutamine': ('EX_gln__L_e', -2)}
    combination_size : int
        Size of metabolite combinations to test (2 for pairs, 3 for triplets, etc.)
    base_metabolites : dict or None
        Base metabolites always present. If None, tests all combinations.
    
    Returns:
    --------
    pd.DataFrame
        Results showing growth rates for each combination
    """
    
    results = []
    metabolite_names = list(metabolites_dict.keys())
    
    print(f"Testing {combination_size}-metabolite combinations...")
    print(f"Total metabolites: {len(metabolite_names)}")
    
    # Generate all combinations
    combos = list(combinations(metabolite_names, combination_size))
    print(f"Number of combinations to test: {len(combos)}\\n")
    
    for combo in tqdm(combos, desc="Testing combinations"):
        with model:
            # CRITICAL: Set up minimal medium first
            setup_minimal_medium(model)
            
            # Reset all other exchange reactions to allow only secretion by default
            essential_rxn_ids = ['EX_h2o_e', 'EX_h_e', 'EX_co2_e', 'EX_pi_e',
                                'EX_so4_e', 'EX_nh4_e', 'EX_ca2_e', 'EX_cl_e',
                                'EX_k_e', 'EX_na1_e', 'EX_mg2_e', 'EX_fe2_e',
                                'EX_fe3_e', 'EX_cu2_e', 'EX_zn2_e', 'EX_mn2_e',
                                'EX_cobalt2_e', 'EX_mobd_e', 'EX_btn_e',
                                'EX_pnto__R_e', 'EX_ribflv_e', 'EX_thm_e',
                                'EX_fol_e', 'EX_ncam_e', 'EX_pydxn_e', 'EX_cbl1_e']
            
            for rxn in model.exchanges:
                if rxn.id.startswith('EX_') and rxn.id not in essential_rxn_ids:
                    # Allow secretion, block uptake by default
                    rxn.bounds = (0, 1000)
            
            # Set base metabolites if provided
            if base_metabolites:
                for met_name, (rxn_id, rate) in base_metabolites.items():
                    if rxn_id in model.reactions:
                        if rate < 0:
                            model.reactions.get_by_id(rxn_id).bounds = (rate, 1000)
                        else:
                            model.reactions.get_by_id(rxn_id).bounds = (0, rate)
            
            # Always allow oxygen
            if 'EX_o2_e' in model.reactions:
                model.reactions.EX_o2_e.bounds = (-20, 1000)
            
            # Set the metabolites in the combination
            active_metabolites = []
            for met_name in combo:
                rxn_id, rate = metabolites_dict[met_name]
                if rxn_id in model.reactions:
                    if rate < 0:  # Uptake
                        model.reactions.get_by_id(rxn_id).bounds = (rate, 1000)
                    else:  # Secretion
                        model.reactions.get_by_id(rxn_id).bounds = (0, rate)
                    active_metabolites.append(met_name)
            
            # Run optimization
            try:
                solution = model.optimize()
                
                if solution.status == 'optimal':
                    growth_rate = solution.objective_value
                    status = 'viable'
                else:
                    growth_rate = 0.0
                    status = solution.status
            except Exception as e:
                growth_rate = 0.0
                status = 'failed'
            
            # Store result
            combo_str = ' + '.join(combo)
            results.append({
                'Combination': combo_str,
                'Metabolites': list(combo),
                'Growth Rate (1/h)': growth_rate,
                'Status': status,
                'Viable': growth_rate > 0.001
            })
    
    # Create DataFrame and sort by growth rate
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Growth Rate (1/h)', ascending=False).reset_index(drop=True)
    
    # Summary statistics
    viable_count = (results_df['Growth Rate (1/h)'] > 0.001).sum()
    print(f"\\n✓ Analysis completed!")
    print(f"Viable combinations: {viable_count}/{len(results_df)}")
    if len(results_df) > 0:
        print(f"Best growth rate: {results_df['Growth Rate (1/h)'].max():.4f} 1/h")
    
    return results_df

In [ ]:
def analyze_metabolite_combinations(model, 
                                   metabolites_dict,
                                   combination_size=2,
                                   base_metabolites=None):
    """
    Analyze the effect of metabolite combinations on growth.
    
    Parameters:
    -----------
    model : cobra.Model
        The iCHO3K metabolic model
    metabolites_dict : dict
        Dictionary mapping metabolite names to exchange reaction IDs and uptake rates
        Example: {'Glucose': ('EX_glc__D_e', -10), 'Glutamine': ('EX_gln__L_e', -2)}
    combination_size : int
        Size of metabolite combinations to test (2 for pairs, 3 for triplets, etc.)
    base_metabolites : dict or None
        Base metabolites always present. If None, tests all combinations.
    
    Returns:
    --------
    pd.DataFrame
        Results showing growth rates for each combination
    """
    
    results = []
    metabolite_names = list(metabolites_dict.keys())
    
    print(f"Testing {combination_size}-metabolite combinations...")
    print(f"Total metabolites: {len(metabolite_names)}")
    
    # Generate all combinations
    combos = list(combinations(metabolite_names, combination_size))
    print(f"Number of combinations to test: {len(combos)}\n")
    
    for combo in tqdm(combos, desc="Testing combinations"):
        with model:
            # Reset all exchange reactions
            for rxn in model.exchanges:
                if rxn.id.startswith('EX_'):
                    # Allow secretion, block uptake by default
                    rxn.bounds = (0, 1000)
            
            # Set base metabolites if provided
            if base_metabolites:
                for met_name, (rxn_id, rate) in base_metabolites.items():
                    if rxn_id in model.reactions:
                        if rate < 0:
                            model.reactions.get_by_id(rxn_id).bounds = (rate, 0)
                        else:
                            model.reactions.get_by_id(rxn_id).bounds = (0, rate)
            
            # Always allow essential metabolites
            essential = {
                'EX_o2_e': (-20, 0),
                'EX_h2o_e': (-1000, 1000),
                'EX_h_e': (-1000, 1000),
                'EX_pi_e': (-10, 1000),
            }
            for rxn_id, bounds in essential.items():
                if rxn_id in model.reactions:
                    model.reactions.get_by_id(rxn_id).bounds = bounds
            
            # Set the metabolites in the combination
            active_metabolites = []
            for met_name in combo:
                rxn_id, rate = metabolites_dict[met_name]
                if rxn_id in model.reactions:
                    if rate < 0:  # Uptake
                        model.reactions.get_by_id(rxn_id).bounds = (rate, 0)
                    else:  # Secretion
                        model.reactions.get_by_id(rxn_id).bounds = (0, rate)
                    active_metabolites.append(met_name)
            
            # Run optimization
            try:
                solution = model.optimize()
                
                if solution.status == 'optimal':
                    growth_rate = solution.objective_value
                    status = 'viable'
                else:
                    growth_rate = 0.0
                    status = solution.status
            except Exception as e:
                growth_rate = 0.0
                status = 'failed'
            
            # Store result
            combo_str = ' + '.join(combo)
            results.append({
                'Combination': combo_str,
                'Metabolites': list(combo),
                'Growth Rate (1/h)': growth_rate,
                'Status': status,
                'Viable': growth_rate > 0.001
            })
    
    # Create DataFrame and sort by growth rate
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Growth Rate (1/h)', ascending=False).reset_index(drop=True)
    
    # Summary statistics
    viable_count = (results_df['Growth Rate (1/h)'] > 0.001).sum()
    print(f"\n✓ Analysis completed!")
    print(f"Viable combinations: {viable_count}/{len(results_df)}")
    print(f"Best growth rate: {results_df['Growth Rate (1/h)'].max():.4f} 1/h")
    
    return results_df

### 7.1 Define Metabolites to Test

In [ ]:
# Define metabolites with their exchange reactions and typical uptake rates
# Format: 'Name': ('exchange_reaction_id', uptake_rate)
# Negative rate = uptake, Positive rate = secretion

test_metabolites = {
    'Glucose': ('EX_glc__D_e', -10.0),
    'Glutamine': ('EX_gln__L_e', -2.0),
    'Serine': ('EX_ser__L_e', -0.5),
    'Glycine': ('EX_gly_e', -0.5),
    'Asparagine': ('EX_asn__L_e', -0.5),
    'Proline': ('EX_pro__L_e', -0.3),
    'Pyruvate': ('EX_pyr_e', -1.0),
    'Alanine': ('EX_ala__L_e', -0.5),
}

# Define base metabolites (always present)
base_metabolites = {
    'Glucose': ('EX_glc__D_e', -10.0),
    'Glutamine': ('EX_gln__L_e', -2.0),
}

print("Test metabolites defined:")
for name, (rxn, rate) in test_metabolites.items():
    print(f"  {name}: {rxn} ({rate} mmol/gDW/h)")

### 7.2 Test Metabolite Pairs

Test all pairwise combinations of metabolites

In [ ]:
# Test all pairs of metabolites
pair_results = analyze_metabolite_combinations(
    model=model,
    metabolites_dict=test_metabolites,
    combination_size=2,
    base_metabolites=None  # Test pairs without base
)

# Display top 10 combinations
print("\nTop 10 metabolite pairs by growth rate:")
print(pair_results.head(10)[['Combination', 'Growth Rate (1/h)', 'Status']])

### 7.3 Test Metabolite Triplets

Test all combinations of 3 metabolites

In [ ]:
# Test triplets
triplet_results = analyze_metabolite_combinations(
    model=model,
    metabolites_dict=test_metabolites,
    combination_size=3,
    base_metabolites=None
)

# Display top 10 combinations
print("\nTop 10 metabolite triplets by growth rate:")
print(triplet_results.head(10)[['Combination', 'Growth Rate (1/h)', 'Status']])

### 7.4 Test Supplements with Base Medium

Test additional metabolites with glucose + glutamine as base

In [ ]:
# Test single supplements added to base medium
supplement_metabolites = {
    'Serine': ('EX_ser__L_e', -0.5),
    'Glycine': ('EX_gly_e', -0.5),
    'Asparagine': ('EX_asn__L_e', -0.5),
    'Proline': ('EX_pro__L_e', -0.3),
    'Pyruvate': ('EX_pyr_e', -1.0),
    'Alanine': ('EX_ala__L_e', -0.5),
}

supplement_results = analyze_metabolite_combinations(
    model=model,
    metabolites_dict=supplement_metabolites,
    combination_size=1,
    base_metabolites=base_metabolites
)

print("\nSupplement effects on growth (with glucose + glutamine base):")
print(supplement_results[['Combination', 'Growth Rate (1/h)', 'Status']])

### 7.5 Visualize Metabolite Combination Results

In [ ]:
def plot_combination_results(results_df, title="Metabolite Combination Analysis", top_n=15):
    """
    Visualize metabolite combination analysis results.
    """
    # Take top N results
    plot_df = results_df.head(top_n).copy()
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Bar chart of growth rates
    colors = ['green' if v else 'red' for v in plot_df['Viable']]
    axes[0].barh(range(len(plot_df)), plot_df['Growth Rate (1/h)'], color=colors, alpha=0.7)
    axes[0].set_yticks(range(len(plot_df)))
    axes[0].set_yticklabels(plot_df['Combination'], fontsize=9)
    axes[0].set_xlabel('Growth Rate (1/h)', fontsize=11)
    axes[0].set_title(f'{title}\nTop {top_n} Combinations', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='x')
    axes[0].invert_yaxis()
    
    # Plot 2: Distribution of growth rates
    axes[1].hist(results_df['Growth Rate (1/h)'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    axes[1].axvline(results_df['Growth Rate (1/h)'].mean(), color='red', linestyle='--', 
                    linewidth=2, label=f'Mean: {results_df["Growth Rate (1/h)"].mean():.4f}')
    axes[1].set_xlabel('Growth Rate (1/h)', fontsize=11)
    axes[1].set_ylabel('Frequency', fontsize=11)
    axes[1].set_title('Distribution of Growth Rates', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot pair results
plot_combination_results(pair_results, "Metabolite Pair Analysis", top_n=15)

# Plot triplet results
plot_combination_results(triplet_results, "Metabolite Triplet Analysis", top_n=15)

## 8. Compare Batch Simulations with Different Conditions

Run multiple batch simulations with different metabolite supplementations

In [ ]:
# Define different conditions to test
conditions = {
    'Base (Glc+Gln)': {
        'EX_glc__D_e': (-10, 0),
        'EX_gln__L_e': (-2, 0),
    },
    'High Glucose': {
        'EX_glc__D_e': (-15, 0),
        'EX_gln__L_e': (-2, 0),
    },
    '+ Serine': {
        'EX_glc__D_e': (-10, 0),
        'EX_gln__L_e': (-2, 0),
        'EX_ser__L_e': (-0.5, 0),
    },
    '+ Pyruvate': {
        'EX_glc__D_e': (-10, 0),
        'EX_gln__L_e': (-2, 0),
        'EX_pyr_e': (-1, 0),
    },
}

# Run simulations for each condition
condition_results = {}

for condition_name, bounds in conditions.items():
    print(f"\n{'='*60}")
    print(f"Testing condition: {condition_name}")
    print(f"{'='*60}")
    
    # Modify model temporarily
    with model:
        # Apply the bounds
        for rxn_id, bound in bounds.items():
            if rxn_id in model.reactions:
                model.reactions.get_by_id(rxn_id).bounds = bound
        
        # Run simulation
        results = simulate_batch_culture(
            model=model,
            days=5,
            initial_biomass=0.2,
            initial_glucose=25.0,
            initial_glutamine=4.0,
            timestep=1.0
        )
        
        condition_results[condition_name] = results

print("\n✓ All condition simulations completed!")

### 8.1 Compare Results Across Conditions

In [ ]:
# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot biomass for all conditions
for condition_name, df in condition_results.items():
    axes[0, 0].plot(df['Time (days)'], df['Biomass (g/L)'], linewidth=2, label=condition_name, marker='o', markersize=3)
axes[0, 0].set_xlabel('Time (days)', fontsize=11)
axes[0, 0].set_ylabel('Biomass (g/L)', fontsize=11)
axes[0, 0].set_title('Biomass Growth - Condition Comparison', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Plot glucose
for condition_name, df in condition_results.items():
    axes[0, 1].plot(df['Time (days)'], df['Glucose (mM)'], linewidth=2, label=condition_name, marker='o', markersize=3)
axes[0, 1].set_xlabel('Time (days)', fontsize=11)
axes[0, 1].set_ylabel('Glucose (mM)', fontsize=11)
axes[0, 1].set_title('Glucose Consumption', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Plot lactate
for condition_name, df in condition_results.items():
    axes[1, 0].plot(df['Time (days)'], df['Lactate (mM)'], linewidth=2, label=condition_name, marker='o', markersize=3)
axes[1, 0].set_xlabel('Time (days)', fontsize=11)
axes[1, 0].set_ylabel('Lactate (mM)', fontsize=11)
axes[1, 0].set_title('Lactate Production', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# Plot growth rate
for condition_name, df in condition_results.items():
    axes[1, 1].plot(df['Time (days)'], df['Growth Rate (1/h)'], linewidth=2, label=condition_name, marker='o', markersize=3)
axes[1, 1].set_xlabel('Time (days)', fontsize=11)
axes[1, 1].set_ylabel('Growth Rate (1/h)', fontsize=11)
axes[1, 1].set_title('Growth Rate Comparison', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
summary_data = []
for condition_name, df in condition_results.items():
    summary_data.append({
        'Condition': condition_name,
        'Final Biomass (g/L)': df['Biomass (g/L)'].iloc[-1],
        'Max Growth Rate (1/h)': df['Growth Rate (1/h)'].max(),
        'Avg Growth Rate (1/h)': df['Growth Rate (1/h)'].mean(),
        'Final Lactate (mM)': df['Lactate (mM)'].iloc[-1]
    })

summary_df = pd.DataFrame(summary_data)
print("\nCondition Comparison Summary:")
print(summary_df.to_string(index=False))

## 9. Export Results

Save all simulation results to CSV files

In [ ]:
# Export batch simulation results
batch_results.to_csv('batch_simulation_results.csv', index=False)
print("✓ Batch simulation results exported to: batch_simulation_results.csv")

# Export metabolite combination results
pair_results.to_csv('metabolite_pair_results.csv', index=False)
print("✓ Metabolite pair results exported to: metabolite_pair_results.csv")

triplet_results.to_csv('metabolite_triplet_results.csv', index=False)
print("✓ Metabolite triplet results exported to: metabolite_triplet_results.csv")

# Export condition comparison
for condition_name, df in condition_results.items():
    filename = f"batch_condition_{condition_name.replace(' ', '_').replace('+', 'plus')}.csv"
    df.to_csv(filename, index=False)
    print(f"✓ Condition '{condition_name}' exported to: {filename}")

print("\n✓ All results exported successfully!")

## 10. Custom Analysis Section

Use this section to run your own custom simulations and analyses

In [ ]:
# Example: Run custom batch simulation with specific parameters

custom_results = simulate_batch_culture(
    model=model,
    days=10,  # Customize number of days
    initial_biomass=0.15,  # Customize initial biomass
    initial_glucose=30.0,  # Customize initial glucose
    initial_glutamine=5.0,  # Customize initial glutamine
    timestep=0.5  # Finer time resolution
)

# Plot custom results
plot_batch_results(custom_results)

In [ ]:
# Example: Test your own metabolite combinations

custom_metabolites = {
    'Leucine': ('EX_leu__L_e', -0.3),
    'Isoleucine': ('EX_ile__L_e', -0.3),
    'Valine': ('EX_val__L_e', -0.3),
    'Lysine': ('EX_lys__L_e', -0.3),
}

custom_combo_results = analyze_metabolite_combinations(
    model=model,
    metabolites_dict=custom_metabolites,
    combination_size=2,
    base_metabolites=base_metabolites
)

print("\nCustom metabolite combination results:")
print(custom_combo_results[['Combination', 'Growth Rate (1/h)', 'Status']])

## Conclusion

This notebook provides tools for:

1. **Batch Simulations**: Simulate CHO cell batch cultures over multiple days with customizable parameters
2. **Metabolite Combinations**: Test the effects of different metabolite combinations on cell growth
3. **Comparative Analysis**: Compare different culture conditions and supplementation strategies

### Key Functions:

- `simulate_batch_culture()`: Run batch culture simulations with time-course dynamics
- `analyze_metabolite_combinations()`: Test metabolite combinations systematically
- `plot_batch_results()`: Visualize batch culture time-course data
- `plot_combination_results()`: Visualize metabolite combination analysis

### Next Steps:

- Customize simulation parameters in Section 10
- Add more metabolites to test
- Integrate experimental data
- Perform flux sampling for uncertainty quantification
- Test gene knockouts and their effects on metabolism

For more information about the iCHO3K model, visit: https://github.com/LewisLabUCSD/iCHO3K